<a href="https://colab.research.google.com/github/ThomasAlbin/Space-Science-With-Python/blob/main/2026/03_SPICE_Cassinis_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPICE Lecture 03: Cassini's Journey & Saturn's Sphere of Influence

In this notebook we follow the **Cassini** spacecraft on its long cruise towards Saturn and answer a single, physically motivated question: *when did Cassini enter the gravitational realm of Saturn?*

To do this we combine the generic planetary ephemeris with the mission's own **reconstructed trajectory kernels** released through the [NAIF PDS archive](https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/):

- **`co_1997288_97318_launch_v1.bsp`** — launch phase (Oct–Nov 1997)
- **`co_1997319_99311_i_cru_v1.bsp`** — inbound cruise (1997–1999)
- **`co_1999312_01066_o_cru_v1.bsp`** — outbound cruise (1999–2001)
- **`041014R_SCPSE_01066_04199.bsp`** — approach & Saturn tour (2001–2004)

Cassini carries the NAIF ID **`-82`**. We will compute its distance to Saturn over time and compare it against Saturn's **sphere of influence (SOI)** — the region where Saturn's gravity dominates over the Sun's:

$$ r_{\mathrm{SOI}} = a \left( \frac{m_{\mathrm{Saturn}}}{m_{\mathrm{Sun}}} \right)^{2/5} $$

where $a$ is the (instantaneous) Saturn–Sun distance used here as the semi-major axis.

In [1]:
import datetime
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spiceypy as spice

plt.style.use('dark_background')
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['font.size'] = 11

In [2]:
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

kernels = {
    # ── Generic kernels ──────────────────────────────────────────────────────
    "naif0012.tls": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls",
    "pck00010.tpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc",
    "gm_de440.tpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de440.tpc",
    "de432s.bsp":   "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de432s.bsp",
    # ── Cassini mission SPKs (NAIF PDS archive) ───────────────────────────────
    "co_1997288_97318_launch_v1.bsp": "https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/co_1997288_97318_launch_v1.bsp",
    "co_1997319_99311_i_cru_v1.bsp":  "https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/co_1997319_99311_i_cru_v1.bsp",
    "co_1999312_01066_o_cru_v1.bsp":  "https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/co_1999312_01066_o_cru_v1.bsp",
    "041014R_SCPSE_01066_04199.bsp":  "https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/041014R_SCPSE_01066_04199.bsp",
}

for name, url in kernels.items():
    path = data_dir / name
    if not path.exists():
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, path)
    else:
        print(f"{name} already exists.")

for name in kernels:
    spice.furnsh(str(data_dir / name))

naif0012.tls already exists.
pck00010.tpc already exists.
gm_de440.tpc already exists.
de432s.bsp already exists.
co_1997288_97318_launch_v1.bsp already exists.
co_1997319_99311_i_cru_v1.bsp already exists.
co_1999312_01066_o_cru_v1.bsp already exists.
041014R_SCPSE_01066_04199.bsp already exists.


In [3]:
# # Alternative path: using curl or wget

# # Leapseconds (LSK)
# !curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls --create-dirs -o ../data/naif0012.tls

# # Planetary constants (PCK) + gravitational parameters
# !curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc --create-dirs -o ../data/pck00010.tpc
# !curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de440.tpc --create-dirs -o ../data/gm_de440.tpc

# # Planetary ephemeris (SPK)
# !curl https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de432s.bsp --create-dirs -o ../data/de432s.bsp

# # Cassini mission SPKs
# !curl https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/co_1997288_97318_launch_v1.bsp --create-dirs -o ../data/co_1997288_97318_launch_v1.bsp
# !curl https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/co_1997319_99311_i_cru_v1.bsp --create-dirs -o ../data/co_1997319_99311_i_cru_v1.bsp
# !curl https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/co_1999312_01066_o_cru_v1.bsp --create-dirs -o ../data/co_1999312_01066_o_cru_v1.bsp
# !curl https://naif.jpl.nasa.gov/pub/naif/pds/data/co-s_j_e_v-spice-6-v1.0/cosp_1000/data/spk/041014R_SCPSE_01066_04199.bsp --create-dirs -o ../data/041014R_SCPSE_01066_04199.bsp

---
## 1. Cassini SPK Coverage

Before any computation we inspect **where in time** Cassini (NAIF ID `-82`) is defined. Each mission kernel covers a different leg of the journey; together they form a continuous ephemeris from launch in 1997 to the end of the reconstructed approach in mid-2004. We use `spice.spkcov()` to read the coverage window of `-82` from every Cassini SPK.

In [ ]:
CASSINI_SPKS = [
    "co_1997288_97318_launch_v1.bsp",
    "co_1997319_99311_i_cru_v1.bsp",
    "co_1999312_01066_o_cru_v1.bsp",
    "041014R_SCPSE_01066_04199.bsp",
]

coverage = []
for name in CASSINI_SPKS:
    cov = spice.spkcov(str(data_dir / name), -82)
    for i in range(spice.wncard(cov)):
        et_start, et_end = spice.wnfetd(cov, i)
        coverage.append({
            'Kernel': name,
            'Coverage Start': spice.et2utc(et_start, 'ISOC', 0),
            'Coverage End': spice.et2utc(et_end,   'ISOC', 0),
            'Span (days)': round((et_end - et_start) / 86400.0, 1),
        })

df_cov = pd.DataFrame(coverage)
df_cov

,Kernel,Coverage Start,Coverage End,Span (days)
0,co_1997288_97318_launch_v1.bsp,1997-10-15T09:26:08,1997-11-14T23:58:57,30.6
1,co_1997319_99311_i_cru_v1.bsp,1997-11-14T23:58:57,1999-07-06T16:00:00,598.7
2,co_1997319_99311_i_cru_v1.bsp,1999-07-06T16:00:00,1999-11-07T23:58:56,124.3
3,co_1999312_01066_o_cru_v1.bsp,1999-11-07T23:58:56,2001-03-07T11:58:56,485.5
4,041014R_SCPSE_01066_04199.bsp,2001-03-07T11:58:56,2004-07-17T11:58:56,1228.0


---
## 2. Saturn's Position & Sphere of Influence

We first need Saturn's distance from the **Solar System Barycenter (SSB)**. We query the state vector of the **Saturn barycenter** (NAIF ID `6`) with `spice.spkgeo()` and take the norm of its position. Following the task definition we use this instantaneous distance as the orbital semi-major axis $a$.

We evaluate it at Saturn Orbit Insertion (1 July 2004) — the epoch most relevant to our SOI question.

In [ ]:
# Reference epoch: Saturn Orbit Insertion
et_ref = spice.str2et('2004-07-01')

# State vector of the Saturn barycenter w.r.t. the Solar System Barycenter (SSB = ID 0)
saturn_state_wrt_ssb, saturn_ssb_lt = spice.spkgeo(targ=6,
                                                    et=et_ref,
                                                    ref='ECLIPJ2000',
                                                    obs=0)

one_au = spice.convrt(1.0, 'AU', 'KM')
a_saturn_km = np.linalg.norm(saturn_state_wrt_ssb[:3])

print("State vector of Saturn w.r.t. SSB at 2004-07-01 (ECLIPJ2000):")
print(f"  Position (km): x={saturn_state_wrt_ssb[0]:.3f}, y={saturn_state_wrt_ssb[1]:.3f}, z={saturn_state_wrt_ssb[2]:.3f}")
print(f"  Velocity (km/s): vx={saturn_state_wrt_ssb[3]:.3f}, vy={saturn_state_wrt_ssb[4]:.3f}, vz={saturn_state_wrt_ssb[5]:.3f}")
print(f"\nSaturn-SSB distance (semi-major axis a): {a_saturn_km:.3f} km ({a_saturn_km / one_au:.3f} AU)")

State vector of Saturn w.r.t. SSB at 2004-07-01 (ECLIPJ2000):
  Position (km): x=-383414639.769, y=1296800967.992, z=-7304737.270
  Velocity (km/s): vx=-9.779, vy=-2.761, vz=0.437

Saturn-SSB distance (semi-major axis a): 1352313904.299 km (9.040 AU)


In [ ]:
# ── Extract gravitational parameters (GM) from the PCK / gm_de440 kernel ─────
# The mass ratio in the SOI equation equals the GM ratio (same gravitational constant G).
gm_sun = spice.bodvrd("SUN", 'GM', 1)[1][0]   # km^3/s^2
gm_saturn = spice.bodvrd("SATURN BARYCENTER", 'GM', 1)[1][0]  # km^3/s^2

print(f"GM of the Sun:    {gm_sun:.5e} km^3/s^2")
print(f"GM of Saturn:     {gm_saturn:.5e} km^3/s^2")
print(f"Mass ratio (Saturn / Sun): {gm_saturn / gm_sun:.6e}")

# ── Sphere of Influence: r = a * (m_saturn / m_sun) ** (2/5) ──────────────────
soi_radius_km = a_saturn_km * (gm_saturn / gm_sun) ** (2.0 / 5.0)

print(f"\nSaturn's Sphere of Influence radius: {soi_radius_km:.3f} km "
      f"({soi_radius_km / one_au:.4f} AU)")

GM of the Sun:    1.32712e+11 km^3/s^2
GM of Saturn:     3.79406e+07 km^3/s^2
Mass ratio (Saturn / Sun): 2.858857e-04

Saturn's Sphere of Influence radius: 51707610.437 km (0.3456 AU)


---
## 3. When Does Cassini Enter Saturn's SOI?

We now march forward in time from **1 January 1998** in **1-day steps**, computing Cassini's distance to Saturn at every step. As soon as the distance drops below the SOI radius we stop the loop: the last two epochs bracket the moment Cassini crossed into the Saturnian system.

In [ ]:
# Cassini's ephemeris ends mid-2004; use that as a hard safety limit for the loop
et_loop_end = spice.str2et('2004-07-17')

et = spice.str2et('1998-01-01')
step = 86400.0   # 1 day in seconds
et_previous = et

while et < et_loop_end:
    # State of Cassini (-82) w.r.t. the Saturn barycenter (6)
    cassini_state_wrt_saturn, _ = spice.spkgeo(targ=-82, et=et, ref='ECLIPJ2000', obs=6)
    distance_km = np.linalg.norm(cassini_state_wrt_saturn[:3])

    if distance_km < soi_radius_km:
        break

    et_previous = et
    et += step

print("Cassini probably entered Saturn's Sphere of Influence between:")
print(f"  {spice.et2utc(et_previous, 'C', 0)}  and  {spice.et2utc(et, 'C', 0)}")
print(f"\nDistance at this step: {distance_km:.3f} km ({distance_km / one_au:.4f} AU)")
print(f"SOI radius:            {soi_radius_km:.3f} km ({soi_radius_km / one_au:.4f} AU)")

Cassini probably entered Saturn's Sphere of Influence between:
  2004 MAR 17 23:59:59  and  2004 MAR 18 23:59:59

Distance at this step: 51540133.194 km (0.3445 AU)
SOI radius:            51707610.437 km (0.3456 AU)


---
## 4. Cross-Check with the SPICE Geometry Finder `gfposc`

The day-stepping loop above only resolves the crossing to within one day. SPICE provides dedicated **geometry finders** that solve for the exact event time. Here we adapt the rectangular-coordinate example to a **spherical** coordinate search: we look for the time interval in which the **`RADIUS`** coordinate of Cassini's position relative to Saturn is **smaller than** the SOI radius.

In [10]:
et_start = spice.str2et('1998-01-01')
et_end   = spice.str2et('2004-07-17')

# Define the search window (confinement window)
cnfine = spice.utils.support_types.SPICEDOUBLE_CELL(2)
spice.wninsd(et_start, et_end, cnfine)

# Geometric "finder" settings and requirements
target = "CASSINI"           # NAIF ID -82
frame  = "ECLIPJ2000"
abcorr = "NONE"
obsrvr = "SATURN BARYCENTER"

crdsys = "SPHERICAL"         # We work in (radius, lon, lat)
coord  = "RADIUS"            # We are searching on the radial distance

relate = "<"                 # distance smaller than ...
refval = soi_radius_km       # ... the Sphere of Influence radius
adjust = 0.0
step   = 86400.0

# Result window
result = spice.utils.support_types.SPICEDOUBLE_CELL(1000)

# Actual geometry finder
spice.gfposc(target,
             frame,
             abcorr,
             obsrvr,
             crdsys,
             coord,
             relate,
             refval,
             adjust,
             step,
             1000,
             cnfine,
             result)

count = spice.wncard(result)
print(f"Found {count} interval(s) where Cassini is inside Saturn's SOI.\n")
for i in range(count):
    interval_start, interval_end = spice.wnfetd(result, i)
    print(f"  Entered SOI: {spice.et2utc(interval_start, 'C', 3)}")
    print(f"  Window ends: {spice.et2utc(interval_end,   'C', 3)}  (= end of search window / kernel coverage)")

Found 1 interval(s) where Cassini is inside Saturn's SOI.

  Entered SOI: 2004 MAR 18 15:19:26.681
  Window ends: 2004 JUL 17 00:00:00.000  (= end of search window / kernel coverage)
